In [1]:
import logging
import numpy as np
import pandas as pd

In [2]:
logger = logging.getLogger('evol.parse')

In [3]:
def stack_plate(n):
    # the plate reader writes elapsed time either as "HH:MM:SS" or, once a full
    # day has passed, as "N days, HH:MM:SS"; to_timedelta parses both and keeps
    # multi-day reads correct (the old manual parser hardcoded a single 24h
    # rollover, collapsing days 2+ onto day 1 and creating duplicate timepoints)
    n['time'] = pd.to_timedelta(n['Time'].astype('str')).dt.total_seconds().astype(float)
    # n['time'] = n['time'] / 3600.0

    n = n.drop(columns=['Time'])

    n = n.set_index('time').stack()
    n.name = 'od600'
    n.index.names = ['time', 'well']
    n = n.reset_index()
    n['row'] = [x[0] for x in n['well'].values]
    n['column'] = [int(x[1:]) for x in n['well'].values]

    n = n.drop(columns=['well'])
    return n

def parse_excel_time_series(infile):
    '''Parse an excel output from the BioTek plate reader

    The expecation is that the OD values are multiple timepoints, from a 384-well
    microplate.
    '''
    m = pd.read_excel(infile, parse_dates=False)
    n = m.dropna()
    n = n.reset_index(drop=True)

    n = n.drop(columns='T° 600')
    # idxs = sorted(n[n['Unnamed: 1'] == 'Time'].index)
    
    # res = []
    # for idx1, idx2 in zip(idxs, idxs[1:]):
    #     t = n.iloc[idx1:idx2]
    #     t.columns = t.iloc[0]
    #     t = t.drop(columns='T° Read 1:600')
    #     t = t.iloc[1:].copy()
    #     t = stack_plate(t)
    #     res.append(t)
    # t = n.iloc[idx2:]
    # t.columns = t.iloc[0]
    # t = t.drop(columns='T° Read 1:600')
    # t = t.iloc[1:].copy()
    n = stack_plate(n)
    # res.append(t)

    # n = pd.concat(res)

    return n.set_index(['row', 'column'])[['time', 'od600']]

In [4]:
n = parse_excel_time_series('rep_1_od.xlsx').reset_index()

n['strain'] = pd.Series(np.nan, index=n.index, dtype=object)
n.loc[n[n['column'] == 1].index,
      'strain'] = 'PL'
n.loc[n[n['column'] == 2].index,
      'strain'] = 'PM'
n.loc[n[n['column'] == 3].index,
      'strain'] = 'LM'
n.loc[n[n['column'].isin([4, 5])].index,
      'strain'] = 'PL+PM'
n.loc[n[n['column'].isin([6, 7])].index,
      'strain'] = 'LM+PL'
n.loc[n[n['column'].isin([8, 9])].index,
      'strain'] = 'LM+PM'
n.loc[n[n['column'].isin([10, 11, 12])].index,
      'strain'] = 'TriComm'

n['concentration'] = 0.
n['plate'] = '96-custom'
n['experiment'] = 'growth_rate_20260720'
n['treatment'] = pd.Series(np.nan, index=n.index, dtype=object)
n.loc[n[n['row'] == 'A'].index,
      'treatment'] = 'M9'
n.loc[n[n['row'] == 'B'].index,
      'treatment'] = 'M9+Phe'
n.loc[n[n['row'] == 'C'].index,
      'treatment'] = 'M9+Met'
n.loc[n[n['row'] == 'D'].index,
      'treatment'] = 'M9+Lys'
n.loc[n[n['row'] == 'E'].index,
      'treatment'] = 'M9+Lys+Phe'
n.loc[n[n['row'] == 'F'].index,
      'treatment'] = 'M9+Met+Phe'
n.loc[n[n['row'] == 'G'].index,
      'treatment'] = 'M9+Lys+Met'
n.loc[n[n['row'] == 'H'].index,
      'treatment'] = 'M9+Lys+Met+Phe'

n = n[n['time'] <= 61 * 3600].copy()

n = n.sort_values(['row', 'column', 'time']).reset_index(drop=True)

n.dropna().to_csv('rep_1.tsv', sep='\t', index=False)

In [5]:
n.groupby(['row', 'column'])['od600'].max().sort_values().tail(20)

row  column
G    5         0.542
     4         0.543
     8         0.549
H    3         0.560
G    10        0.579
A    5         0.579
F    8         0.592
     12        0.625
H    5         0.635
     4         0.671
     9         0.701
A    10        0.711
H    8         0.712
G    12        0.774
A    11        0.816
H    2         0.821
     10        0.824
     11        0.833
A    12        0.912
H    12        0.986
Name: od600, dtype: float64

In [6]:
n = parse_excel_time_series('rep_2_od.xlsx').reset_index()

n['strain'] = pd.Series(np.nan, index=n.index, dtype=object)
n.loc[n[n['column'] == 1].index,
      'strain'] = 'PL'
n.loc[n[n['column'] == 2].index,
      'strain'] = 'PM'
n.loc[n[n['column'] == 3].index,
      'strain'] = 'LM'
n.loc[n[n['column'].isin([4, 5])].index,
      'strain'] = 'PL+PM'
n.loc[n[n['column'].isin([6, 7])].index,
      'strain'] = 'LM+PL'
n.loc[n[n['column'].isin([8, 9])].index,
      'strain'] = 'LM+PM'
n.loc[n[n['column'].isin([10, 11, 12])].index,
      'strain'] = 'TriComm'

n['concentration'] = 0.
n['plate'] = '96-custom'
n['experiment'] = 'growth_rate_20260727'
n['treatment'] = pd.Series(np.nan, index=n.index, dtype=object)
n.loc[n[n['row'] == 'A'].index,
      'treatment'] = 'M9'
n.loc[n[n['row'] == 'B'].index,
      'treatment'] = 'M9+Phe'
n.loc[n[n['row'] == 'C'].index,
      'treatment'] = 'M9+Met'
n.loc[n[n['row'] == 'D'].index,
      'treatment'] = 'M9+Lys'
n.loc[n[n['row'] == 'E'].index,
      'treatment'] = 'M9+Lys+Phe'
n.loc[n[n['row'] == 'F'].index,
      'treatment'] = 'M9+Met+Phe'
n.loc[n[n['row'] == 'G'].index,
      'treatment'] = 'M9+Lys+Met'
n.loc[n[n['row'] == 'H'].index,
      'treatment'] = 'M9+Lys+Met+Phe'

n = n[n['time'] <= 61 * 3600].copy()

n = n.sort_values(['row', 'column', 'time']).reset_index(drop=True)

n.dropna().to_csv('rep_2.tsv', sep='\t', index=False)

In [7]:
n.groupby(['row', 'column'])['od600'].max().sort_values().tail(20)

row  column
G    8         0.569
H    7         0.591
G    10        0.597
A    11        0.629
     5         0.635
G    11        0.659
F    12        0.661
A    4         0.672
     10        0.677
H    9         0.688
     8         0.691
     1         0.695
     4         0.719
     5         0.737
     2         0.748
     10        0.788
G    12        0.841
A    12        0.871
H    11        0.887
     12        1.194
Name: od600, dtype: float64